In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import itertools

import pandas as pd
import umap
import numpy as np
import joblib

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

import plotly.graph_objects as go
import plotly.express as px
from plotly.offline import plot

# UMAP + Gower + DBSCAN Clustering Pipeline

Pipeline para análisis de clustering tridimensional sobre datos mixtos, combinando métricas de Gower, reducción de dimensionalidad con UMAP y agrupamiento con DBSCAN.

## 📋 Descripción general
El flujo de trabajo incluye:
- Preprocesamiento de datos con escalado y filtrado de características
- Cálculo de distancias para datos mixtos usando métrica de Gower
- Reducción a 3 dimensiones con UMAP (optimizando hiperparámetros)
- Agrupamiento con DBSCAN (selección automática de parámetros)
- Evaluación mediante silhouette score
- Visualización interactiva 3D con Plotly

## 🔍 Metodología

### 1. Carga y preprocesamiento
Los datos se cargan desde un archivo CSV y se normalizan al rango [0,1]. Se eliminan variables constantes mediante umbral de varianza.

### 2. Distancia de Gower
Se calcula una matriz de similitud adaptada para datos mixtos (numéricos y categóricos).

### 3. Optimización de UMAP
Se explora sistemáticamente combinaciones de:
- Vecindarios (n_neighbors)
- Distancias mínimas (min_dist)
- Dispersión (spread)

### 4. Clustering con DBSCAN
Para cada embedding UMAP óptimo, se ajusta DBSCAN buscando la mejor combinación de:
- Radio de vecindad (eps)
- Mínimas muestras (min_samples)

### 5. Evaluación y visualización
Los resultados se evalúan con silhouette score y se generan visualizaciones 3D interactivas donde cada punto representa un registro coloreado por cluster.

## 🏆 Mejor resultado obtenido
- Configuración UMAP: 100 vecinos, distancia mínima 0.01, dispersión 0.5
- Configuración DBSCAN: eps=10, min_samples=3
- Silhouette score: 0.930
- Clusters identificados: 5

## 📂 Estructura de salida
Los resultados incluyen:
- Archivos CSV con embeddings y etiquetas
- Modelos serializados (UMAP y DBSCAN)
- Visualizaciones HTML interactivas

## ⚙️ Requisitos
- Python 3.8+
- Dependencias: pandas, numpy, scikit-learn, umap-learn, gower, plotly, joblib

Instalación con:
`pip install -r requirements.txt`

In [2]:
# 1) Carga y definición de X
path = "data/lista_perpetrador.csv"
df   = pd.read_csv(path)
X    = df.values

# 2) Escalado a [0,1]
scaler      = MinMaxScaler(feature_range=(0, 1))
X_scaled    = scaler.fit_transform(X)

# 3) Eliminar dimensiones de varianza cero
vt       = VarianceThreshold(threshold=0.0)
X_bin    = vt.fit_transform(X_scaled)


In [3]:
import gower

# Calcula la matriz de distancia Gower (n_samples × n_samples)
D_gower = gower.gower_matrix(X_bin)


In [4]:
# Parámetros de grid (igual que antes)
param_grid = {
    'n_neighbors': [3, 5, 15, 50, 100],
    'min_dist':    [0.01, 0.1, 0.2, 0.5, 1, 2, 10],
    'spread':      [0.3, 0.5, 2, 3, 5, 15],
    # Fíjate: obligamos a usar precomputed
    'metric':      ['precomputed']
}
n_components = 3  # embedding 3D

# DBSCAN grid igual
dbscan_eps_list         = param_grid['min_dist']
dbscan_min_samples_list = param_grid['n_neighbors']

# Carpeta de salida
output_dir = "umap_perpetrador_dbscan_html"
os.makedirs(output_dir, exist_ok=True)

i = 0
for n_n, m_d, sp, metric in itertools.product(
        param_grid['n_neighbors'],
        param_grid['min_dist'],
        param_grid['spread'],
        param_grid['metric']
    ):
    if m_d > sp:
        continue
    i += 1

    # 5.1) UMAP con precomputed
    reducer = umap.UMAP(
        n_neighbors = n_n,
        min_dist     = m_d,
        spread       = sp,
        n_components = n_components,
        metric       = 'precomputed',  # aquí
        random_state = 42
    )
    embedding = reducer.fit_transform(D_gower)   # PASAMOS LA MATRIZ D_gower

    # 5.2) Grid-search DBSCAN
    best_score  = -1
    best_labels = None
    best_params = {}
    for eps in dbscan_eps_list:
        for min_s in dbscan_min_samples_list:
            db     = DBSCAN(eps=eps, min_samples=min_s)
            labels = db.fit_predict(embedding)
            if len(set(labels)) > 1:
                score = silhouette_score(embedding, labels)
            else:
                score = -1
            if score > best_score:
                best_score  = score
                best_labels = labels
                best_params = {'eps': eps, 'min_samples': min_s}

    # 5.3) Guardar resultados
    df_out = pd.DataFrame({
        'umap_1': embedding[:, 0],
        'umap_2': embedding[:, 1],
        'umap_3': embedding[:, 2],
        'cluster': best_labels
    })
    df_out.to_csv(os.path.join(output_dir, f"comb_{i}_results.csv"),
                  index=False)

    # 5.4) Guardar modelos
    joblib.dump(reducer,
                os.path.join(output_dir, f"comb_{i}_umap_model.pkl"))
    joblib.dump(DBSCAN(**best_params).fit(embedding),
                os.path.join(output_dir, f"comb_{i}_dbscan_model.pkl"))

    # 5.5) Visualización 3D interactiva (igual que antes)
    fig = go.Figure()
    palette = px.colors.qualitative.Set1
    for idx, label in enumerate(np.unique(best_labels)):
        mask = (best_labels == label)
        name = 'Noise' if label == -1 else f'Cluster {label}'
        fig.add_trace(go.Scatter3d(
            x=embedding[mask, 0],
            y=embedding[mask, 1],
            z=embedding[mask, 2],
            mode='markers',
            marker=dict(size=3, opacity=0.7,
                        color=palette[idx % len(palette)]),
            name=name
        ))

    fig.update_layout(
        title=(
            f"UMAP-Gower (n_n={n_n}, min_d={m_d}, spread={sp})<br>"
            f"DBSCAN eps={best_params['eps']}, "
            f"min_samples={best_params['min_samples']} "
            f"(silhouette={best_score:.3f})"
        ),
        scene=dict(
            xaxis_title='UMAP 1',
            yaxis_title='UMAP 2',
            zaxis_title='UMAP 3'
        ),
        legend=dict(itemsizing='constant',
                    bgcolor='rgba(0,0,0,0)', borderwidth=1),
        margin=dict(l=0, r=0, b=0, t=80)
    )
    plot(fig,
         filename=os.path.join(output_dir, f"comb_{i}_dbscan.html"),
         auto_open=False)

    print(f"[Iter {i}] UMAP-Gower: n_n={n_n}, min_d={m_d}, spread={sp} "
          f"-> DBSCAN eps={best_params['eps']}, "
          f"min_s={best_params['min_samples']} "
          f"(silhouette={best_score:.3f})")

print(f"Total iteraciones: {i}. Revisa '{output_dir}/'")

[Iter 1] UMAP-Gower: n_n=3, min_d=0.01, spread=0.3 -> DBSCAN eps=2, min_s=50 (silhouette=0.433)
[Iter 2] UMAP-Gower: n_n=3, min_d=0.01, spread=0.5 -> DBSCAN eps=2, min_s=15 (silhouette=0.502)
[Iter 3] UMAP-Gower: n_n=3, min_d=0.01, spread=2 -> DBSCAN eps=2, min_s=15 (silhouette=0.677)
[Iter 4] UMAP-Gower: n_n=3, min_d=0.01, spread=3 -> DBSCAN eps=2, min_s=15 (silhouette=0.657)
[Iter 5] UMAP-Gower: n_n=3, min_d=0.01, spread=5 -> DBSCAN eps=10, min_s=50 (silhouette=0.687)
[Iter 6] UMAP-Gower: n_n=3, min_d=0.01, spread=15 -> DBSCAN eps=10, min_s=100 (silhouette=0.529)
[Iter 7] UMAP-Gower: n_n=3, min_d=0.1, spread=0.3 -> DBSCAN eps=1, min_s=15 (silhouette=0.378)
[Iter 8] UMAP-Gower: n_n=3, min_d=0.1, spread=0.5 -> DBSCAN eps=2, min_s=15 (silhouette=0.575)
[Iter 9] UMAP-Gower: n_n=3, min_d=0.1, spread=2 -> DBSCAN eps=2, min_s=15 (silhouette=0.691)
[Iter 10] UMAP-Gower: n_n=3, min_d=0.1, spread=3 -> DBSCAN eps=2, min_s=15 (silhouette=0.629)
[Iter 11] UMAP-Gower: n_n=3, min_d=0.1, spread=5 ->

In [5]:
import os
import pandas as pd

def summarize_csv_clusters(directory: str) -> pd.DataFrame:
    """
    Recorre todos los archivos .csv en el directorio dado y devuelve un DataFrame
    con, para cada archivo:
      - file: nombre del archivo
      - count_minus_one: cantidad de valores -1 en la columna 'cluster'
      - num_categories: número de categorías únicas en la columna 'cluster'
      - error (opcional): mensaje de error si no se pudo procesar el archivo
    """
    records = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(".csv"):
            path = os.path.join(directory, filename)
            try:
                df = pd.read_csv(path)
                if 'cluster' in df.columns:
                    count_minus_one = (df['cluster'] == -1).sum()
                    num_categories = df['cluster'].nunique()
                else:
                    count_minus_one = None
                    num_categories = None
                records.append({
                    'file': filename,
                    'count_minus_one': count_minus_one,
                    'num_categories': num_categories
                })
            except Exception as e:
                records.append({
                    'file': filename,
                    'count_minus_one': None,
                    'num_categories': None,
                    'error': str(e)
                })
    return pd.DataFrame(records)

# Ejemplo de uso:
path = r"umap_perpetrador_dbscan_html"
summary_df = summarize_csv_clusters(path)
summary_df

,file,count_minus_one,num_categories
0,comb_100_results.csv,0,9
1,comb_101_results.csv,0,10
2,comb_102_results.csv,552,4
3,comb_103_results.csv,192,5
4,comb_104_results.csv,0,6
...,...,...,...
155,comb_96_results.csv,400,7
156,comb_97_results.csv,0,7
157,comb_98_results.csv,0,6
158,comb_99_results.csv,0,7


In [8]:
import re 

def load_iter_silhouette(filepath: str) -> pd.DataFrame:
    """
    Carga un archivo de texto donde cada línea tiene el formato:
      [Iter i] ... (silhouette=x.y)
    y devuelve un DataFrame con dos columnas:
      - iter: el número i
      - silhouette: el valor x.y
    """
    # Expresión regular para capturar el número de iter y el valor de silhouette
    pattern = re.compile(r'\[Iter\s+(\d+)\].*?\(silhouette=([0-9]*\.?[0-9]+)\)')
    
    registros = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for línea in f:
            m = pattern.search(línea)
            if m:
                iter_num = int(m.group(1))
                sil = float(m.group(2))
                registros.append({'iter': iter_num, 'silhouette': sil})
    
    # Construimos el DataFrame
    df = pd.DataFrame(registros, columns=['iter', 'silhouette'])
    return df

In [10]:
df_silhouette = load_iter_silhouette('cluster_perpetrador.txt')
df_silhouette.shape

(160, 2)

In [16]:
perpetrador_cluster_metrics = summary_df.join(df_silhouette).drop(columns="iter")
perpetrador_cluster_metrics.to_csv(r"data/metric_umap_DBSCAN_perpretador.csv",index=False)

In [20]:
perpetrador_cluster_metrics.sort_values("silhouette", ascending=False).head(35)

,file,count_minus_one,num_categories,silhouette
130,comb_73_results.csv,0,16,0.930
129,comb_72_results.csv,0,15,0.930
131,comb_74_results.csv,0,12,0.927
136,comb_79_results.csv,0,15,0.926
97,comb_43_results.csv,158,10,0.924
143,comb_85_results.csv,0,14,0.924
142,comb_84_results.csv,0,13,0.922
137,comb_7_results.csv,62,13,0.921
105,comb_50_results.csv,3549,5,0.920
147,comb_89_results.csv,0,12,0.919
